# TB Pipeline — Ablation Studies

Runs the six ablations needed for the conference paper. Each ablation reuses the same held-out 560-image evaluation split as `eval_moe.ipynb` and writes its own component/system CSVs under `/kaggle/working/ablations/<variant_name>/`.

**Ablations:**
| ID | Name | What changes | Variants |
|---|---|---|---|
| A  | Adaptive Otsu vs Fixed | Otsu floor/ceil bounds | clamped[0.5,0.8] (current), pure (0,1), fixed 0.5, fixed 0.7 |
| C  | Cavity threshold sweep | sigmoid cutoff for Expert 2 | 0.85, 0.75, 0.65 (current), 0.55 |
| E  | Fusion mode | logit-space vs probability-space | weighted_logit (current), weighted_prob |
| D  | Routing variants | how the gate weights experts | soft (current), top1, uniform, force-consolidation |
| F  | Routing temperature | softmax sharpness at inference | 0.25, 0.5 (current), 1.0, 2.0 |
| B  | Domain context | gate input includes TXV ctx | on (current); off requires retrained gate (caveat noted) |

**Cost:** ~12 min/run × ~16 runs ≈ 3.5 hrs on a single T4.

## Datasets to attach
Same as `eval_moe.ipynb`: `iahmedhabib/montgomery`, `iahmedhabib/shehzhenn`, `usmanshams/tbx-11`, `iahmedhabib/medsam-vit-b`, your trained-checkpoints dataset, and `organizations/nih-chest-xrays`.

In [ ]:
# Cell 1: Clone repo and install deps
import os, subprocess, sys
from pathlib import Path

REPO_DIR = Path('/kaggle/working/repo')
if not REPO_DIR.exists():
    subprocess.run(
        ['git', 'clone', '-b', 'cleaned-repo',
         'https://github.com/mabdullahi7780/dl-project-codebase.git', str(REPO_DIR)],
        check=True
    )
os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

subprocess.run(['pip', 'install', '-r', 'requirements.txt', '-q'], check=True)
print('Repo ready:', REPO_DIR)

In [ ]:
# Cell 2: Paths
from pathlib import Path
import torch

KAGGLE_INPUT = Path('/kaggle/input')
MONTGOMERY_ROOT  = KAGGLE_INPUT / 'datasets/iahmedhabib/montgomery/montgomery'
SHENZHEN_ROOT    = KAGGLE_INPUT / 'datasets/iahmedhabib/shehzhenn/shenzhen'
TBX11K_ROOT      = KAGGLE_INPUT / 'datasets/usmanshams/tbx-11/TBX11K'
NIH_ROOT         = KAGGLE_INPUT / 'datasets/organizations/nih-chest-xrays/data'
CHECKPOINTS_ROOT = KAGGLE_INPUT / 'datasets/iahmedhabib/tb-pipeline-checkpoints'
MEDSAM_CKPT      = KAGGLE_INPUT / 'datasets/iahmedhabib/medsam-vit-b/medsam_vit_b.pth'

for label, p in {'montgomery':MONTGOMERY_ROOT,'shenzhen':SHENZHEN_ROOT,'tbx11k':TBX11K_ROOT,'medsam':MEDSAM_CKPT}.items():
    print(f'  {label:<11}: {p}  {"OK" if p.exists() else "MISSING"}')
nih_available = NIH_ROOT.exists()
print(f'  nih_cxr14  : {"available" if nih_available else "NOT attached"}')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'\nDevice: {device}')

In [ ]:
# Cell 3: Write base configs + ablation runner helper
import yaml, json
from pathlib import Path

ABL_DIR = Path('/kaggle/working/ablations')
ABL_DIR.mkdir(parents=True, exist_ok=True)

def _find(*candidates):
    for c in candidates:
        if Path(c).is_file(): return str(c)
    return None

c1_adapter  = _find(CHECKPOINTS_ROOT/'component1/component1_adapters.safetensors',
                    REPO_DIR/'checkpoints/component1/component1_adapters.safetensors')
c4_decoder  = _find(CHECKPOINTS_ROOT/'component4/component4_mask_decoder.pt',
                    REPO_DIR/'checkpoints/component4/component4_mask_decoder.pt')
c2_routing  = _find(CHECKPOINTS_ROOT/'component2/component2_routing_head.pt',
                    REPO_DIR/'checkpoints/component2/component2_routing_head.pt')
moe_ckpt    = _find(CHECKPOINTS_ROOT/'component_moe/moe_best.pt',
                    REPO_DIR/'checkpoints/component_moe/moe_best.pt')
critic_ckpt = _find(CHECKPOINTS_ROOT/'component_moe/boundary_critic.pt',
                    REPO_DIR/'checkpoints/component_moe/boundary_critic.pt')

# Optional: a no-domain-ctx gate checkpoint for Ablation B (off variant).
# If you have not retrained, leave this None — Ablation B will be skipped.
moe_ckpt_no_domain_ctx = _find(CHECKPOINTS_ROOT/'component_moe/moe_best_nodomctx.pt')

paths_cfg = {
    'project_root': str(REPO_DIR),
    'external_data_root': str(KAGGLE_INPUT),
    'datasets': {
        'montgomery': str(MONTGOMERY_ROOT), 'shenzhen': str(SHENZHEN_ROOT),
        'tbx11k': str(TBX11K_ROOT),
        'nih_cxr14': str(NIH_ROOT) if nih_available else str(ABL_DIR/'_missing_nih'),
    },
    'artifacts': {'notebook_cache': str(ABL_DIR/'cache'), 'processed': str(ABL_DIR/'processed'), 'reports': str(ABL_DIR/'reports')},
}
paths_file = ABL_DIR/'paths.kaggle.yaml'
paths_file.write_text(yaml.dump(paths_cfg))

with (REPO_DIR/'configs/moe.yaml').open() as f:
    moe_cfg = yaml.safe_load(f) or {}
moe_cfg.setdefault('runtime', {})['device'] = device
moe_cfg.setdefault('component1', {}).update({'backend':'auto','checkpoint_path':str(MEDSAM_CKPT),'adapter_path':c1_adapter})
moe_cfg.setdefault('component2', {}).update({'backend':'auto','routing_head_path':c2_routing})
moe_cfg.setdefault('component4', {}).update({'backend':'auto','checkpoint_path':str(MEDSAM_CKPT),'model_type':'vit_b','decoder_checkpoint_path':c4_decoder})
moe_cfg.setdefault('moe', {}).update({'enabled':True,'checkpoint_path':moe_ckpt})
moe_cfg.setdefault('component7_moe', {})['boundary_critic_checkpoint'] = critic_ckpt

moe_file = ABL_DIR/'moe.kaggle.yaml'
moe_file.write_text(yaml.dump(moe_cfg))
print('Configs written under', ABL_DIR)
print('  paths_file :', paths_file)
print('  moe_file   :', moe_file)
print('  moe_ckpt_no_domain_ctx :', moe_ckpt_no_domain_ctx or 'NOT FOUND (Ablation B will skip)')

In [ ]:
# Cell 4: Ablation runner helpers
import json, pandas as pd
from pathlib import Path
from src.evaluation.moe_eval_ablation import AblationOverrides, run_ablation

# How many images per domain. 200 = full holdout (matches eval_moe.ipynb).
# Set to 30 for a quick smoke test (~2 min/run).
LIMIT_PER_DOMAIN = 200

def _row_value(df, metric, dataset):
    sub = df[(df['metric']==metric) & (df['dataset']==dataset)]
    if sub.empty: return None
    v = sub['value'].iloc[0]
    try: return float(v)
    except (ValueError, TypeError): return None

def collect_summary(variant_dir: Path, name: str, ablation_id: str) -> dict:
    sys_csv = variant_dir / 'moe_system.csv'
    cmp_csv = variant_dir / 'moe_components.csv'
    if not sys_csv.exists():
        return {'ablation': ablation_id, 'variant': name, 'error': 'no system csv'}
    sys_df = pd.read_csv(sys_csv)
    cmp_df = pd.read_csv(cmp_csv)
    return {
        'ablation': ablation_id, 'variant': name,
        'shenzhen_timika_auroc'  : _row_value(sys_df, 'timika_auroc',      'shenzhen'),
        'montgomery_timika_auroc': _row_value(sys_df, 'timika_auroc',      'montgomery'),
        'tbx11k_timika_auroc'    : _row_value(sys_df, 'timika_auroc',      'tbx11k'),
        'shenzhen_alp_mean'      : _row_value(sys_df, 'alp_mean',          'shenzhen'),
        'montgomery_alp_mean'    : _row_value(sys_df, 'alp_mean',          'montgomery'),
        'tbx11k_alp_mean'        : _row_value(sys_df, 'alp_mean',          'tbx11k'),
        'nih_alp_mean'           : _row_value(sys_df, 'alp_mean',          'nih_cxr14'),
        'cavity_rate_shenzhen'   : _row_value(sys_df, 'cavity_detection_rate', 'shenzhen'),
        'cavity_rate_montgomery' : _row_value(sys_df, 'cavity_detection_rate', 'montgomery'),
        'cavity_rate_tbx11k'     : _row_value(sys_df, 'cavity_detection_rate', 'tbx11k'),
        'tb_head_shenzhen'       : _row_value(sys_df, 'tb_head_auroc',     'shenzhen'),
        'lung_dice_montgomery'   : _row_value(cmp_df, 'c4_lung_dice',      'montgomery'),
        'lung_dice_shenzhen'     : _row_value(cmp_df, 'c4_lung_dice',      'shenzhen'),
        'moe_lesion_dice_tbx11k' : _row_value(cmp_df, 'moe_lesion_dice',   'tbx11k'),
    }

all_results = []

def run_and_collect(ov: AblationOverrides, ablation_id: str):
    summary = run_ablation(
        ov,
        moe_config_path=moe_file,
        paths_config_path=paths_file,
        output_dir=ABL_DIR,
        limit_per_domain=LIMIT_PER_DOMAIN,
        tbx_list_name='all_trainval.txt',
        repo_root=REPO_DIR,
    )
    row = collect_summary(ABL_DIR/ov.name, ov.name, ablation_id)
    all_results.append(row)
    print(f'\n>>> {ablation_id}/{ov.name}: shenzhen_timika={row.get("shenzhen_timika_auroc")}, nih_alp={row.get("nih_alp_mean")}')
    return summary

print('Helpers ready. LIMIT_PER_DOMAIN =', LIMIT_PER_DOMAIN)

## Ablation A — Adaptive Otsu vs Fixed Threshold

Tests the binarisation strategy applied to the fused lesion probability map. The current default clamps Otsu's threshold to `[0.5, 0.8]` to prevent over-firing on healthy images. Variants:

- `A1_clamped_0.5_0.8` — current default
- `A2_pure_otsu` — no clamping (floor=0.0, ceil=1.0)
- `A3_fixed_0.5` — naive fixed threshold (floor=ceil=0.5)
- `A4_fixed_0.7` — conservative fixed (floor=ceil=0.7)

In [ ]:
# Ablation A — Otsu mode
for ov in [
    AblationOverrides(name='A1_clamped_0.5_0.8'),  # default — no overrides
    AblationOverrides(name='A2_pure_otsu',  otsu_floor=0.0, otsu_ceil=1.0),
    AblationOverrides(name='A3_fixed_0.5',  otsu_floor=0.5, otsu_ceil=0.5),
    AblationOverrides(name='A4_fixed_0.7',  otsu_floor=0.7, otsu_ceil=0.7),
]:
    run_and_collect(ov, 'A_otsu')

## Ablation C — Cavity Threshold Sweep

The cavity flag (+40 Timika points) fires when Expert 2's sigmoid output exceeds this threshold. The original 0.85 produced 0% cavity detection on Shenzhen TB+ cases; the current 0.65 yields 69%. We sweep to justify the operating point.

In [ ]:
# Ablation C — Cavity threshold
for thr in [0.85, 0.75, 0.65, 0.55]:
    ov = AblationOverrides(name=f'C_cav_{thr}', cavity_threshold=thr)
    run_and_collect(ov, 'C_cavity_threshold')

## Ablation E — Logit-Space vs Probability-Space Fusion

`weighted_logit` (default) sums expert logits weighted by routing weights, then sigmoids. `weighted_prob` sigmoids each expert first, then averages probabilities. The latter compresses high-confidence predictions toward 0.5.

In [ ]:
# Ablation E — Fusion mode
for mode in ['weighted_logit', 'weighted_prob']:
    ov = AblationOverrides(name=f'E_{mode}', fusion_mode=mode)
    run_and_collect(ov, 'E_fusion_mode')

## Ablation D — Routing Variants (no retraining)

Tests alternative routing decisions at inference time:

- `D1_soft_moe` — current default (softmax-weighted mixture of all 4 experts)
- `D2_top1_hard` — only the argmax expert contributes (hard MoE)
- `D3_uniform` — all experts contribute equally (no learned routing)
- `D4_force_consolidation` — single-expert proxy: all weight on Consolidation (Expert 0)

D4 is our "single decoder" baseline — answers "does the MoE actually help vs just one trained decoder?"

In [ ]:
# Ablation D — Routing variants
for ov in [
    AblationOverrides(name='D1_soft_moe'),  # default
    AblationOverrides(name='D2_top1_hard',  routing_override='top1'),
    AblationOverrides(name='D3_uniform',    routing_override='uniform'),
    AblationOverrides(name='D4_force_consolidation', routing_override='force_expert_0'),
]:
    run_and_collect(ov, 'D_routing_variant')

## Ablation F — Routing Temperature (inference-only sweep)

Sweep softmax temperature applied to gate logits at inference. The gate was trained at τ=0.5; this measures how sensitive routing is to the chosen sharpness. **Caveat:** strict ablation requires retraining at each τ, but for a small sweep range an inference-only sweep is informative.

In [ ]:
# Ablation F — Routing temperature (inference-only)
for tau in [0.25, 0.5, 1.0, 2.0]:
    ov = AblationOverrides(name=f'F_tau_{tau}', routing_temperature=tau)
    run_and_collect(ov, 'F_routing_temperature')

## Ablation B — Domain Context On/Off

Tests whether the gate benefits from receiving the TXV domain context vector. **This ablation requires a separately retrained gate without domain ctx**, because flipping the flag at inference produces silent garbage (the gate's domain-projection layer expects an input). If `moe_ckpt_no_domain_ctx` is not found, this cell skips.

In [ ]:
# Ablation B — Domain context on/off (requires retrained gate for off-variant)
if moe_ckpt_no_domain_ctx is None:
    print('SKIP — No retrained no-domain-ctx checkpoint found at',
          CHECKPOINTS_ROOT/'component_moe/moe_best_nodomctx.pt')
    print('To run this ablation, retrain Phase 2 with use_domain_ctx=False and upload moe_best_nodomctx.pt')
else:
    # On variant = current default
    run_and_collect(AblationOverrides(name='B1_domain_ctx_on'), 'B_domain_ctx')
    # Off variant — retrained gate without domain ctx
    run_and_collect(
        AblationOverrides(
            name='B2_domain_ctx_off',
            use_domain_ctx=False,
            moe_checkpoint_path=moe_ckpt_no_domain_ctx,
        ),
        'B_domain_ctx',
    )

## Aggregated Ablation Results

Builds a single master CSV with one row per variant and the headline columns the paper needs.

In [ ]:
# Build master ablation table + per-ablation tables for paper
import pandas as pd

master_df = pd.DataFrame(all_results)
master_csv = ABL_DIR / 'ablation_master.csv'
master_df.to_csv(master_csv, index=False)
print(f'Wrote {master_csv}')
print(master_df.to_string(index=False))

# Per-ablation tables — these are what the paper imports
for ablation_id in sorted({r['ablation'] for r in all_results}):
    sub = master_df[master_df['ablation']==ablation_id]
    out = ABL_DIR / f'table_{ablation_id}.csv'
    sub.to_csv(out, index=False)
    print(f'  {ablation_id} -> {out}')

In [ ]:
# Quick sanity plots — Shenzhen Timika AUROC across all ablation variants
import matplotlib.pyplot as plt

ok = master_df.dropna(subset=['shenzhen_timika_auroc']).copy()
if not ok.empty:
    fig, ax = plt.subplots(figsize=(11, 4))
    bars = ax.bar(ok['variant'], ok['shenzhen_timika_auroc'], color='#2196F3', alpha=0.85)
    ax.bar_label(bars, fmt='%.3f', padding=2, fontsize=8, rotation=0)
    ax.axhline(0.5, color='red', linestyle='--', linewidth=1, label='Random')
    ax.set_ylim(0, 1.0)
    ax.set_ylabel('Shenzhen Timika AUROC')
    ax.set_title('All Ablation Variants — Shenzhen Timika AUROC')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.savefig(ABL_DIR / 'ablation_overview.png', dpi=150)
    plt.show()

In [ ]:
# Final: list all output files for download
from pathlib import Path
print('=' * 60)
print('ABLATION OUTPUTS — for download from /kaggle/working/ablations/')
print('=' * 60)
for p in sorted(ABL_DIR.rglob('*')):
    if p.is_file() and p.suffix in ('.csv', '.png', '.json'):
        print(f'  {p.relative_to(ABL_DIR)}')
print('\nDownload the entire /kaggle/working/ablations/ directory via the Kaggle Output tab.')